# Trial P1 — True BCF-1 + causal QA wiring

Recomputes exact A0 and frozen SigLIP2 S1 `G1_COVERAGE_COARSE`, then applies A0-Top5-protected equal RRF60. No legacy `P0_COARSE`, GT, ASR v1.2 access, parameter sweep, or production promotion.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
REPO_URL=os.environ.get('AIC_REPO_URL','https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF=os.environ.get('AIC_REPO_REF','TRIAGEEG'); ANCHOR='f9a8153a58be24db533f6455e70641e2951e3f75'
REPO_DIR=Path(os.environ.get('AIC_REPO_DIR','/kaggle/working/AIC2026_TeamPTK_SGU')); REFRESH_REPO=os.environ.get('AIC_REFRESH_REPO','0')=='1'
TRIAL_MODE=os.environ.get('AIC_TRIAL_MODE','TRIAL_BCF1_SAFE').upper()
TRIAL_INPUT=Path(os.environ.get('AIC_TRIAL_P1_ROOT','/kaggle/input/datasets/irthn1311/thunghiem-bo-de-thi'))
DATA_INPUT=Path(os.environ.get('AIC_DATA_ROOT','/kaggle/input/datasets/nadkli/dataset-aic'))
STAGE1_INPUT=Path(os.environ.get('AIC_STAGE1_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle'))
STAGE1B_INPUT=Path(os.environ.get('AIC_STAGE1B_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports'))
STAGE1E_INPUT=Path(os.environ.get('AIC_STAGE1E_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze'))
CLIP_INPUT=Path(os.environ.get('AIC_CLIP_ROOT','/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32'))
OPUS_INPUT=Path(os.environ.get('AIC_OPUS_ROOT','/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en'))
SIGLIP_INPUT=Path(os.environ.get('AIC_SIGLIP2_ROOT','/kaggle/input/datasets/irthn1311/aic2026-siglip2-base-patch16-224'))
INDEX_INPUT=Path(os.environ.get('AIC_SCA1_INDEX_ROOT','/kaggle/input/datasets/irthn1311/triage-eg-sca1-siglip2-index-v01'))
BCF1_FREEZE_INPUT=Path(os.environ.get('AIC_BCF1_FREEZE_ROOT','/kaggle/input/datasets/irthn1311/bcf1-preparation-freeze-2026-08-18'))
QWEN_INPUT=Path(os.environ.get('AIC_QWEN_ROOT','/kaggle/input/datasets/irthn1311/fs1-qwen2-5-vl-3b-instruct-asset'))
OUTPUT_ROOT=Path('/kaggle/working/trial_p1_TRUE_BCF1_artifacts'); WORK_ROOT=Path('/kaggle/working/trial_p1_true_bcf1_work')
SUBMISSION_ZIP=Path('/kaggle/working/trial_p1_TRUE_BCF1_submission.zip'); BUNDLE_ZIP=Path('/kaggle/working/trial_p1_TRUE_BCF1_bundle.zip')
if TRIAL_MODE not in {'TRIAL_BCF1_SAFE','TRIAL_TRIAGEEG_PREP'}: raise RuntimeError('Unsupported Trial mode; silent fallback is forbidden')
for target in (OUTPUT_ROOT,WORK_ROOT):
    if target.exists(): shutil.rmtree(target)
for target in (SUBMISSION_ZIP,BUNDLE_ZIP): target.unlink(missing_ok=True)
print({'mode':TRIAL_MODE,'required_inputs':{'official_trial_package':str(TRIAL_INPUT),'raw_dataset':str(DATA_INPUT),'stage1_exact_index':str(STAGE1_INPUT),'stage1b_contract':str(STAGE1B_INPUT),'stage1e_language':str(STAGE1E_INPUT),'openai_clip':str(CLIP_INPUT),'opus_mt':str(OPUS_INPUT),'siglip2_asset':str(SIGLIP_INPUT),'exact_prebuilt_siglip2_index':str(INDEX_INPUT),'bcf1_freeze':str(BCF1_FREEZE_INPUT),'qwen_asset':str(QWEN_INPUT)},'internet_required':'ONLY_FOR_GIT_CLONE_OR_EXPLICIT_REFRESH','model_download_required':False,'asr_v12_required':False,'output_submission':str(SUBMISSION_ZIP),'output_bundle':str(BUNDLE_ZIP)})


In [ ]:
def gr(*args,cwd=None): return subprocess.run(['git',*args],cwd=cwd,capture_output=True,text=True,check=False)
def git(*args,cwd=None):
    result=gr(*args,cwd=cwd)
    if result.returncode: raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()
if REPO_DIR.exists() and not (REPO_DIR/'.git').is_dir() and any(REPO_DIR.iterdir()): raise RuntimeError(f'{REPO_DIR} is not a Git checkout')
if not (REPO_DIR/'.git').is_dir(): REPO_DIR.parent.mkdir(parents=True,exist_ok=True); git('clone','--filter=blob:none','--no-checkout',REPO_URL,str(REPO_DIR))
target=None
if not REFRESH_REPO:
    for candidate in (REPO_REF,f'origin/{REPO_REF}'):
        probe=gr('rev-parse','--verify',f'{candidate}^{{commit}}',cwd=REPO_DIR)
        if probe.returncode==0: target=probe.stdout.strip(); break
if target is None: git('fetch','--no-tags','origin',REPO_REF,cwd=REPO_DIR); target='FETCH_HEAD'
git('checkout','--detach',target,cwd=REPO_DIR); HEAD=git('rev-parse','HEAD',cwd=REPO_DIR)
if gr('cat-file','-e',f'{ANCHOR}^{{commit}}',cwd=REPO_DIR).returncode: git('fetch','--no-tags','origin',REPO_REF,cwd=REPO_DIR)
if gr('merge-base','--is-ancestor',ANCHOR,HEAD,cwd=REPO_DIR).returncode: raise RuntimeError(f'Trial anchor {ANCHOR} is not an ancestor of {HEAD}')
if not (REPO_DIR/'src/triage_eg/trial_p1/true_bcf1.py').is_file(): raise RuntimeError('Resolved ref lacks True BCF1 Trial runner')
sys.path.insert(0,str(REPO_DIR/'src'))
print({'source_ref':REPO_REF,'HEAD':HEAD,'anchor_is_ancestor':True,'git_status':git('status','--short',cwd=REPO_DIR) or 'CLEAN'})


In [ ]:
def mounts(hint):
    names={hint.name,hint.name.replace('_','-'),hint.name.replace('-','_')}
    aliases={'thunghiem-bo-de-thi':{'THUNGHIEM-bo-de-thi'},'dataset-aic':{'Dataset_AIC2026'},'bcf1-preparation-freeze-2026-08-18':{'BCF1_PREPARATION_FREEZE_2026-08-18'}}
    names.update(aliases.get(hint.name,set())); candidates=[hint]
    for name in names: candidates.extend((Path('/kaggle/input')/name,Path('/kaggle/input/datasets/irthn1311')/name,Path('/kaggle/input/datasets/nadkli')/name))
    return sorted({path.resolve() for path in candidates if path.exists()})
def mount(hint):
    found=mounts(hint)
    if len(found)!=1: raise RuntimeError(f'Expected exactly one mount for {hint}; found {found}')
    return found[0]
def bounded_dirs(root,max_depth=5,max_directories=4096):
    queue=[(Path(root),0)]; visited=0
    while queue:
        current,depth=queue.pop(0)
        if not current.is_dir(): continue
        visited+=1
        if visited>max_directories: raise RuntimeError(f'Input discovery exceeded {max_directories} directories below {root}')
        yield current
        if depth<max_depth: queue.extend((child,depth+1) for child in sorted(current.iterdir()) if child.is_dir() and not child.is_symlink())
def marker_root(root,marker,optional=False):
    marker=Path(marker); found=[]
    for path in bounded_dirs(root):
        if (path/marker).is_file(): found.append(path.resolve())
    found=sorted(set(found))
    if optional and not found: return None
    if len(found)!=1: raise RuntimeError(f'Expected exactly one root with {marker} below {root}; found {found}')
    return found[0]
TRIAL_MOUNT=mount(TRIAL_INPUT); trial_zips=sorted(TRIAL_MOUNT.rglob('THUNGHIEM-bo-de-thi.zip'))
if len(trial_zips)==1: TRIAL_ZIP=trial_zips[0]; TRIAL_SOURCE='ORIGINAL_ZIP'
elif not trial_zips:
    txts=sorted(TRIAL_MOUNT.rglob('query-p1-*-*.txt')); parents={p.parent.resolve() for p in txts}
    if len(txts)!=24 or len(parents)!=1: raise RuntimeError(f'Expanded Trial package contract failed: {len(txts)}/{parents}')
    TRIAL_ZIP=WORK_ROOT/'THUNGHIEM-bo-de-thi.zip'; TRIAL_ZIP.parent.mkdir(parents=True,exist_ok=True)
    with ZipFile(TRIAL_ZIP,'w',ZIP_DEFLATED) as archive:
        for path in txts: archive.write(path,path.name)
    TRIAL_SOURCE='KAGGLE_EXPANDED_PACKAGE_REPACKED_BYTE_EXACT'
else: raise RuntimeError(f'Ambiguous Trial ZIPs: {trial_zips}')
DATASET_MOUNT=mount(DATA_INPUT); STAGE1_MOUNT=mount(STAGE1_INPUT); STAGE1B_MOUNT=mount(STAGE1B_INPUT); STAGE1E_MOUNT=mount(STAGE1E_INPUT); CLIP_MOUNT=mount(CLIP_INPUT); OPUS_MOUNT=mount(OPUS_INPUT); SIGLIP_MOUNT=mount(SIGLIP_INPUT); INDEX_MOUNT=mount(INDEX_INPUT); BCF1_FREEZE_MOUNT=mount(BCF1_FREEZE_INPUT); QWEN_MOUNT=mount(QWEN_INPUT)
DATASET_ROOTS=sorted({p.resolve() for p in bounded_dirs(DATASET_MOUNT) if (p/'map-keyframes-aic25-b1/map-keyframes').is_dir() and any(p.glob('Videos_*/video'))})
if len(DATASET_ROOTS)!=1: raise RuntimeError(f'Raw dataset discovery failed: {DATASET_ROOTS}')
DATASET_ROOT=DATASET_ROOTS[0]
print({'trial_zip':str(TRIAL_ZIP),'trial_source':TRIAL_SOURCE,'raw':str(DATASET_ROOT),'stage1':str(STAGE1_MOUNT),'stage1b':str(STAGE1B_MOUNT),'stage1e':str(STAGE1E_MOUNT),'clip':str(CLIP_MOUNT),'opus':str(OPUS_MOUNT),'siglip2':str(SIGLIP_MOUNT),'index':str(INDEX_MOUNT),'bcf1_freeze':str(BCF1_FREEZE_MOUNT),'qwen':str(QWEN_MOUNT),'asr_v12_touched':False})


In [ ]:
from triage_eg.diagnostics.bcf1_protected_late_fusion import load_preparation_freeze, validate_frozen_index
from triage_eg.diagnostics.bcf1_protected_late_fusion.contracts import INDEX_ZIP_SHA256
from triage_eg.diagnostics.sca1_siglip2_complementarity import validate_offline_asset
from triage_eg.fs1.contracts import QWEN_REVISION
from triage_eg.retrieval.stage1b.inputs import resolve_stage1_root
from triage_eg.retrieval.stage1d.inputs import resolve_input_root
from aic2026_eval.io import sha256_file, write_json, write_jsonl
STAGE1_ROOT=resolve_stage1_root(STAGE1_MOUNT,search_root=None,materialize_root=WORK_ROOT/'stage1')
STAGE1B_ROOT,_=resolve_input_root(STAGE1B_MOUNT,required=('stage1b_summary.json','encoder/selected_encoder_contract.json','encoder/runtime_adapter_manifest.json'),materialize_root=WORK_ROOT/'stage1b',search_root=None,archive_keyword='stage1b')
STAGE1E_ROOT,_=resolve_input_root(STAGE1E_MOUNT,required=('stage1e_summary.json','language_path_contract.json'),materialize_root=WORK_ROOT/'stage1e',search_root=None,archive_keyword='stage1e')
CLIP_ROOT,_=resolve_input_root(CLIP_MOUNT,required=('checkpoint/ViT-B-32.pt','manifests/asset_manifest.json'),materialize_root=WORK_ROOT/'clip',search_root=None,archive_keyword='clip')
OPUS_ROOT,_=resolve_input_root(OPUS_MOUNT,required=('model/config.json','manifests/asset_manifest.json'),materialize_root=WORK_ROOT/'opus',search_root=None,archive_keyword='opus')
SIGLIP_ROOT,_=resolve_input_root(SIGLIP_MOUNT,required=('model/model.safetensors','manifests/asset_manifest.json'),materialize_root=WORK_ROOT/'siglip2',search_root=None,archive_keyword='siglip2')
INDEX_ROOT=marker_root(INDEX_MOUNT,'index/siglip2_vectors.f16.npy',optional=True); index_zips=sorted(INDEX_MOUNT.rglob('triage_eg_sca1_siglip2_index_v01.zip'))
INDEX_ZIP=index_zips[0] if len(index_zips)==1 else None
if INDEX_ROOT is None:
    if INDEX_ZIP is None or sha256_file(INDEX_ZIP)!=INDEX_ZIP_SHA256: raise RuntimeError('TRUE_BCF1_UNAVAILABLE: exact frozen SigLIP2 index unresolved')
    extracted=WORK_ROOT/'siglip2_index'; extracted.mkdir(parents=True,exist_ok=True)
    with ZipFile(INDEX_ZIP) as archive: archive.extractall(extracted)
    INDEX_ROOT=marker_root(extracted,'index/siglip2_vectors.f16.npy')
BCF1_FREEZE_ROOT=marker_root(BCF1_FREEZE_MOUNT,'bcf1_preparation/decision_context.json',optional=True)
if BCF1_FREEZE_ROOT is None:
    freeze_zips=sorted(BCF1_FREEZE_MOUNT.rglob('BCF1_PREPARATION_FREEZE_2026-08-18.zip'))
    if len(freeze_zips)!=1: raise RuntimeError('TRUE_BCF1_UNAVAILABLE: BCF1 preparation freeze unresolved')
    BCF1_FREEZE_SOURCE=freeze_zips[0]
else: BCF1_FREEZE_SOURCE=BCF1_FREEZE_ROOT
PREPARATION=load_preparation_freeze(BCF1_FREEZE_SOURCE); SIGLIP_ASSET_VALIDATION=validate_offline_asset(SIGLIP_ROOT); INDEX_VALIDATION=validate_frozen_index(INDEX_ROOT,stage1_root=STAGE1_ROOT,index_zip=INDEX_ZIP)
qwen_configs=sorted(QWEN_MOUNT.rglob('config.json'))
if len(qwen_configs)!=1: raise RuntimeError(f'Qwen local-only asset discovery failed: {qwen_configs}')
QWEN_MODEL_ROOT=qwen_configs[0].parent; qwen_config=json.loads(qwen_configs[0].read_text())
QWEN_VALIDATION={'status':'READY_NOT_EXECUTED_WITHOUT_BOUNDED_EVIDENCE','model_id':'Qwen/Qwen2.5-VL-3B-Instruct','exact_revision':QWEN_REVISION,'model_root':str(QWEN_MODEL_ROOT),'config_model_type':qwen_config.get('model_type'),'asr_status':'ASR_PENDING'}
print({'bcf1_freeze':PREPARATION.validation,'siglip2_asset':SIGLIP_ASSET_VALIDATION,'siglip2_index':INDEX_VALIDATION,'qwen_executor':QWEN_VALIDATION,'TRUE_BCF1_UNAVAILABLE':False})


In [ ]:
from triage_eg.trial_p1 import compile_queries, parse_trial_zip
MANIFEST=parse_trial_zip(TRIAL_ZIP); COMPILED=compile_queries(MANIFEST); QUERIES=[plan['team_query'] for plan in COMPILED]
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True); write_json(OUTPUT_ROOT/'trial_p1_query_manifest.json',MANIFEST); write_jsonl(OUTPUT_ROOT/'trial_p1_query_plans_v2.jsonl',COMPILED)
types={plan['query_id']:plan['answer_type'] for plan in COMPILED if plan['task']=='QA'}
assert types=={'query-p1-15-qa':'LOCATION_NAME','query-p1-19-qa':'QUOTE_OR_VISIBLE_TEXT','query-p1-22-qa':'TITLE'}
trake={row['query_id']:row['raw_event_labels'] for row in MANIFEST['queries'] if row['task']=='TRAKE'}; assert trake['query-p1-18-trake']==['E1','E2','E2','E4']
test_env=dict(os.environ); test_env['PYTHONPATH']=os.pathsep.join([str(REPO_DIR/'src'),test_env.get('PYTHONPATH','')]); test_env['AIC_TRIAL_P1_ZIP']=str(TRIAL_ZIP)
test=subprocess.run([sys.executable,'-m','pytest','tests/unit/trial_p1','tests/unit/bcf1_protected_late_fusion','tests/unit/sca1_siglip2_complementarity','tests/unit/e2e1','tests/unit/e2eg1','-q'],cwd=REPO_DIR,env=test_env,capture_output=True,text=True)
if test.returncode: raise RuntimeError(test.stdout+'\n'+test.stderr)
print({'manifest':{'total':24,'KIS':18,'QA':3,'TRAKE':3},'qa_types':types,'trake_raw_labels':trake,'tests':test.stdout.splitlines()[-1],'GT_OPENED':False,'ASR_V12_TOUCHED':False})


In [ ]:
from triage_eg.diagnostics.sca1_siglip2_complementarity import Siglip2ExactBackend, Siglip2GroundingPipeline, Siglip2OfflineEncoder
from triage_eg.e2eg1 import SafeCoveragePipeline
from triage_eg.retrieval.stage1b.adapters.openai_clip_official import materialize_kaggle_expanded_tokenizer, resolve_official_asset_paths
from triage_eg.retrieval.stage2 import OperationalRetrievalRuntime, config_from_yaml
from triage_eg.trial_p1 import run_true_bcf1, write_report_and_bundle
paths=resolve_official_asset_paths(CLIP_ROOT); clip_source,_=materialize_kaggle_expanded_tokenizer(paths.source_root,WORK_ROOT/'shared_openai_clip_source'); os.environ['AIC_OPENAI_CLIP_SOURCE_ROOT']=str(clip_source)
def runtime(name):
    config=config_from_yaml(REPO_DIR/'configs/retrieval/stage2_operational_runtime_gpu.yaml',stage1_root=STAGE1_ROOT,stage1b_root=STAGE1B_ROOT,stage1e_root=STAGE1E_ROOT,clip_asset_root=CLIP_ROOT,translator_asset_root=OPUS_ROOT,output_root=WORK_ROOT/f'runtime_{name}',stage1d_config=REPO_DIR/'configs/retrieval/stage1d_translation_ablation.yaml',build_git_commit=HEAD)
    return OperationalRetrievalRuntime(config).load()
A0_RUNTIME=runtime('a0'); S1_RUNTIME=runtime('s1'); SIGLIP_ENCODER=Siglip2OfflineEncoder(SIGLIP_ROOT,device='auto',batch_size=int(os.environ.get('AIC_SIGLIP2_BATCH_SIZE','64'))).load(); SIGLIP_BACKEND=Siglip2ExactBackend(INDEX_ROOT,stage1_root=STAGE1_ROOT)
A0_PIPELINE=SafeCoveragePipeline(A0_RUNTIME,DATASET_ROOT); S1_PIPELINE=Siglip2GroundingPipeline(S1_RUNTIME,DATASET_ROOT,grounding_encoder=SIGLIP_ENCODER,grounding_backend=SIGLIP_BACKEND)
RESULT=run_true_bcf1(A0_PIPELINE,S1_PIPELINE,COMPILED,OUTPUT_ROOT,SUBMISSION_ZIP,mode=TRIAL_MODE)
print({'result':RESULT,'legacy_P0_COARSE_used':False,'GT_OPENED':False,'ASR_V12_TOUCHED':False})


In [ ]:
PROVENANCE={'HEAD':HEAD,'source_ref':REPO_REF,'trial_source':TRIAL_SOURCE,'resolved_inputs':{'trial_zip':str(TRIAL_ZIP),'raw':str(DATASET_ROOT),'stage1':str(STAGE1_ROOT),'stage1b':str(STAGE1B_ROOT),'stage1e':str(STAGE1E_ROOT),'clip':str(CLIP_ROOT),'opus':str(OPUS_ROOT),'siglip2_asset':str(SIGLIP_ROOT),'siglip2_index':str(INDEX_ROOT),'bcf1_freeze':str(BCF1_FREEZE_SOURCE),'qwen':str(QWEN_MODEL_ROOT)},'siglip2_asset_validation':SIGLIP_ASSET_VALIDATION,'siglip2_index_validation':INDEX_VALIDATION,'qwen_validation':QWEN_VALIDATION,'GT_OPENED':False,'ASR_V12_TOUCHED':False,'automatic_production_promotion':False}
write_json(OUTPUT_ROOT/'run_provenance.json',PROVENANCE); BUNDLE=write_report_and_bundle(OUTPUT_ROOT,BUNDLE_ZIP,RESULT,provenance=PROVENANCE)
A0_PIPELINE.close(); S1_PIPELINE.close(); SIGLIP_ENCODER.close()
print({'TRUE_BCF1_SUBMISSION_VALIDATOR':RESULT['submission_validation'],'QA_COMPILER_CAUSAL_GATE':RESULT['qa_compiler_causal_gate'],'QA_GARBAGE_ANSWER_GATE':RESULT['qa_garbage_answer_gate'],'DOWNLOAD_SUBMISSION':str(SUBMISSION_ZIP),'DOWNLOAD_BUNDLE':str(BUNDLE),'GT_OPENED':False,'ASR_V12_TOUCHED':False,'STOP':True})
